# STARK-Amazon Data Exploration — SKB

Inspects product nodes, graph edges, and the BM25 text corpus.
**Requires ~19 GB RAM — run on GWDG HPC only, not on Mac M2.**
See `01a_data_exploration_qa.ipynb` for split and query inspection (Mac-safe).

## 1. Load QA dataset and SKB

First run downloads to `~/.cache/huggingface/` — subsequent runs load from cache.

In [ ]:
from stark_qa import load_qa, load_skb

qa_dataset = load_qa('amazon')
skb = load_skb('amazon')

print(f'QA dataset size: {len(qa_dataset)}')
print(f'SKB size: {len(skb)}')

In [ ]:
# Set up splits and grab a reference node for the inspection cells below
idx_split = qa_dataset.get_idx_split()
test_ids = idx_split['test']

_, answer_ids, _ = qa_dataset[test_ids[0]]
node_id = answer_ids[0]
print(f'Reference node_id for inspection: {node_id}')

## 2. Inspect a product node (candidate card)

In [ ]:
node = skb.get_node_by_id(node_id)
print(f'Node ID: {node_id}')
print(node)

In [ ]:
# What attributes / fields does a node have?
if hasattr(node, '__dict__'):
    print(list(vars(node).keys()))
elif isinstance(node, dict):
    print(list(node.keys()))

## 3. Inspect graph edges (relations used in Step A)

The five relation types Step A decomposes into:
`has_brand`, `has_category`, `has_color`, `also_bought`, `also_viewed`

In [ ]:
# What relation types exist in the SKB?
if hasattr(skb, 'relation_types'):
    print(skb.relation_types)
elif hasattr(skb, 'edge_types'):
    print(skb.edge_types)
else:
    print(type(skb))
    print([attr for attr in dir(skb) if not attr.startswith('_')])

In [ ]:
# Sample edges for the first answer node
target_relations = ['has_brand', 'has_category', 'has_color', 'also_bought', 'also_viewed']

for rel in target_relations:
    try:
        neighbors = skb.get_neighbor_nodes(node_id, rel)
        print(f'{rel:15s}: {neighbors[:5]}')
    except Exception as e:
        print(f'{rel:15s}: ERROR — {e}')

## 4. Verify the text corpus used for BM25 (Step B)

BM25 needs a text representation of each node. The SKB provides this via `get_doc_info`.

In [ ]:
# Check what text retrieval methods the SKB exposes
print([attr for attr in dir(skb) if 'doc' in attr.lower() or 'text' in attr.lower() or 'corpus' in attr.lower()])

In [ ]:
# Get the text document for the first answer node — this is what BM25 indexes
try:
    doc = skb.get_doc_info(node_id, add_rel=True, compact=True)
    print(doc[:1000])
except Exception as e:
    print(f'get_doc_info failed: {e}')
    for method in ['get_doc', 'get_text', 'node_to_text']:
        if hasattr(skb, method):
            print(f'Found method: {method}')